In [ ]:
%pip install gspread gspread-pandas

In [2]:
import pandas as pd
import gspread
from google.oauth2.service_account import Credentials

CREDENTIALS_FILE = "credentials.json"

TARGET_SHEET_ID = "11ObENuqk8yowgr9TFI7JB-AB3XXNYLu1bU4db-KGcmE"
TARGET_TAB_NAME  = "Combined_ETL"
sources = {
    "CallCenter": {
        "url": "https://docs.google.com/spreadsheets/d/14YHLOv9-U5FdtEzXDMcCAHpy5GNVdsPNSZ9sGlPUSXY/edit?usp=sharing",
        "sheet": "Call Center"
    },
    "AccountMgmt": {
        "url": "https://docs.google.com/spreadsheets/d/1leHVfleZ3tjxkgDv3CzgAZmDe0KtmFBfSAUMLkJtGdk/edit?usp=sharing",
        "sheet": "Account Mangement"
    },
    "Sales": {
        "url": "https://docs.google.com/spreadsheets/d/1QkByTeUwH1m8u4uauZ576XwSCJ2zuE_0aVaKgqWndCQ/edit?usp=sharing",
        "sheet": "Sales"
    },
    "Financials": {
        "url": "https://docs.google.com/spreadsheets/d/1VsdWDZqQQNXkip1S4K_PsdyNLWKuIAIpW7XROtm7qUY/edit?usp=sharing",
        "sheet": "Financials"
    }
}

scopes = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]
creds = Credentials.from_service_account_file(CREDENTIALS_FILE, scopes=scopes)
gc = gspread.authorize(creds)
=
target_sh = gc.open_by_key(TARGET_SHEET_ID)

dfs = []
month_order_map = {
    'January':1, 'February':2, 'March':3, 'April':4, 'May':5, 'June':6,
    'July':7, 'August':8, 'September':9, 'October':10, 'November':11, 'December':12
}

for source, info in sources.items():
    sh = gc.open_by_url(info["url"])
    ws = sh.worksheet(info["sheet"])
    raw_data = ws.get_all_records()
    if not raw_data:
        print(f"Warning: {source} sheet is empty!")
        continue

    df = pd.DataFrame(raw_data)
    df["Month"] = df["Month"].astype(str).str.strip().str.title()

    rename_dict = {col: f"{source}_{col}" for col in df.columns if col != "Month"}
    df = df.rename(columns=rename_dict)

    df["__month_order"] = df["Month"].map(month_order_map).fillna(99)

    dfs.append(df)
    print(f"Loaded {source} — {len(df)} rows")

if not dfs:
    print("No data loaded. Check sheet names / access.")
    exit()

merged = dfs[0]
for df in dfs[1:]:
    merged = merged.merge(
        df,
        on=["Month", "__month_order"],
        how="outer",
        suffixes=("", "_dup")
    )

merged = merged.sort_values("__month_order").drop(columns="__month_order").reset_index(drop=True)

merged = merged.loc[:, ~merged.columns.str.endswith('_dup')]

print(f"\nFinal table: {len(merged)} rows × {len(merged.columns)} columns")

try:
    worksheet = target_sh.worksheet(TARGET_TAB_NAME)
    print(f"Updating existing tab: {TARGET_TAB_NAME}")
except gspread.exceptions.WorksheetNotFound:
    worksheet = target_sh.add_worksheet(title=TARGET_TAB_NAME, rows=2000, cols=300)
    print(f"Created new tab: {TARGET_TAB_NAME}")

worksheet.clear()
headers = merged.columns.tolist()
values = merged.fillna("").values.tolist()

worksheet.update("A1", [headers] + values)
worksheet.format("A1:Z1", {"textFormat": {"bold": True}})

print("\n Done!")
print(f"Combined data saved here: {target_sh.url}")
print(f"Tab: {TARGET_TAB_NAME}")

✓ Loaded CallCenter — 1 rows
✓ Loaded AccountMgmt — 1 rows
✓ Loaded Sales — 1 rows
✓ Loaded Financials — 2 rows

Final table: 2 rows × 24 columns
→ Updating existing tab: Combined_ETL


/tmp/ipykernel_2214/1468245976.py:104: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  worksheet.update("A1", [headers] + values)



🎉 Done!
Combined data saved here: https://docs.google.com/spreadsheets/d/11ObENuqk8yowgr9TFI7JB-AB3XXNYLu1bU4db-KGcmE
Tab: Combined_ETL
